In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import scanpy.external as sce
import scipy
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import pandas as pd
import PyComplexHeatmap as pch
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import pickle
from itertools import combinations
from sklearn.decomposition import PCA

In [ ]:
qs = torch.load('results/spatial/qs.pt',map_location='cpu',weights_only = False)
h  = torch.load('results/spatial/h.pt',map_location='cpu',weights_only = False)
z  = torch.load('results/spatial/zs.pt',map_location='cpu',weights_only = False)
Cs  = torch.load('results/spatial/Cs.pt',map_location='cpu',weights_only = False)

In [ ]:
GCBC,NBC_MBC,FDC,epi,CD4T,mye = [np.argmax(q, axis=1) for q in qs]

In [ ]:
GCBC_label, NBC_MBC_label, FDC_label, epithelial_label, CD4_T_label, myeloid_label = [
    [str(x) for x in np.argmax(q.detach().numpy(), axis=1)] for q in qs
]
qs = [x.detach().numpy() for x in qs]
labels = [GCBC_label, NBC_MBC_label, FDC_label, epithelial_label, CD4_T_label, myeloid_label]

In [ ]:
adata = an.AnnData(h[5].detach().numpy())
#adata.obs['confidence'] = qs[3][np.arange(len(qs[3])), qs[3].argmax(1)]
adata.obs['cluster'] = myeloid_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color=['cluster'],title='UMAP of myeloid Clusters in Latent Space',save="mye_cluster.png")

In [ ]:
pairs = [(0,1), (1,2), (2,3), (3,4), (4,5), (5,0), (0,2), (0,3), (0,4), (1,3), (1,4), (1,5), (2,4), (2,5), (3,5)]
def normalize(A, B):
    C = (A + B.T) / 2
    return (C - C.min()) / (C.max() - C.min())
C = [normalize(Cs[i][j], Cs[j][i]) for i,j in pairs]
'''
C_new = []
for c in C:
    c_new = np.zeros_like(c)         
    max_idx = np.unravel_index(np.argmax(c), c.shape)
    c_new[max_idx] = c[max_idx]    
    C_new.append(c_new)
'''
C_new = []
for c in C:
    c_new = np.where(c > 0.5, c, 0)
    #c_new[c_new <= ] = 0  
    C_new.append(c_new)
    



In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(C[7], fmt="g", cmap='Blues')
plt.title("Matrix Heatmap")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.show()

In [ ]:
g = torch.load('data/spatial/graph/intra/myeloid.pt',weights_only = False)
df = pd.DataFrame(myeloid_label, columns=['cluster'])
order = df.sort_values('cluster').index.tolist()

data_sorted = pd.DataFrame(g).iloc[order, order].reset_index(drop=True)
data_sorted.columns = range(len(data_sorted.columns)) 
df_sorted = df.iloc[order].reset_index(drop=True)

row_ha = pch.HeatmapAnnotation(
    cluster=pch.anno_simple(df_sorted.cluster, add_text=False, legend=False, cmap='tab20'),
    axis=0
)
col_ha = pch.HeatmapAnnotation(
    cluster=pch.anno_simple(df_sorted.cluster, add_text=False, legend=False, cmap='tab20'),
    axis=1
)

plt.figure(figsize=(7, 6))
cm = pch.ClusterMapPlotter(
    data=data_sorted,
    top_annotation=col_ha,
    left_annotation=row_ha,
    row_cluster=False,
    col_cluster=False,
    cmap='Reds',
    vmin=0,
    vmax=1,
    row_names_side='left',
    rasterized=True,
    label = 'correlation'
    
)
plt.suptitle('myeloid graph clustering', fontsize=14, y=0.95)
plt.savefig("clustering result.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
torch.vstack(list(z)).shape

In [ ]:
df = pd.DataFrame(['GCBC']*6+['NBC_MBC']*6+['FDC']*6+['epi']*6+['CD4T']*6+['mye']*6,columns=['cell_type'])
df['cluster'] =[0,1,2,3,4,5]*6


col_ha = pch.HeatmapAnnotation(
    cell_type=pch.anno_simple(df.cell_type, add_text=False, legend=True, cmap='tab20'),
    cluster = pch.anno_simple(df.cluster, add_text =False, legend=True, cmap='tab20'), 
    axis=1
)
plt.figure(figsize=(10.5, 9))
cm = pch.ClusterMapPlotter(
    data=pd.DataFrame(torch.vstack(list(z)).detach().numpy()).T,
    top_annotation=col_ha,
    #left_annotation=row_ha,
    #row_split=df.cluster,
    #row_split=6,
    col_cluster=True,
    col_dendrogram=True, 
    #row_dendrogram_size=2.5,
    #col_cluster=True,
    cmap='Reds',
    vmin=0,
    vmax=1,
    row_names_side='left',
    rasterized=True,
    label='AUC'
)
plt.show()

In [ ]:
path = "marker.csv"
cts = ['GCBC','NBC_MBC','FDC','epithelial','CD4_T','myeloid'] 
df = pd.read_csv(path)

In [ ]:
GCBC = df['GCBC'].dropna()
NBC_MBC = df['NBC_MBC'].dropna()
FDC = df['FDC'].dropna()
epi = df['epithelial'].dropna()
CD4T = df['CD4_T'].dropna()
mye = df['myeloid'].dropna()

In [ ]:
coor = pd.read_csv("data/spatial/coordinates.csv",index_col=0)
stged_ct1 = pd.read_csv(f'data/test/stged/epithelial.csv',index_col=0)
spots = stged_ct1.columns
pos = (
    coor.loc[coor.index.intersection(spots)]
        #.loc[spots][["row", "col"]]
)[["row", "col"]]

pos = pos.values
pos

In [ ]:
def moranR_weights(pos, l=0.65):

    pos = np.array(pos, dtype=np.int32)
    
    diff = pos[:, np.newaxis, :] - pos[np.newaxis, :, :]
    d = np.sum(diff**2, axis=-1)
    d = np.sqrt(d)
    
    n = len(pos)
    w = np.exp(-d**2 / (2 * l**2))
    np.fill_diagonal(w, 0)
    W = np.sum(w)
    weight = (n / W) * w

    return weight


def moranR(x, y, w):

    l_s, l_x = x.shape
    l_y = y.shape[1]
    n = w.shape[0]
    
    x = np.array(x)  # (n_features_x, n_spots)
    y = np.array(y)  # (n_features_y, n_spots)
    
    x_mean = np.mean(x, axis=1, keepdims=True)
    y_mean = np.mean(y, axis=1, keepdims=True)
    x_diff = x - x_mean
    y_diff = y - y_mean
    
    numerator = np.einsum('pi,lj,ij->pl', x_diff, y_diff, w)
    denominator = np.einsum('i,j->ij', 
                           np.sqrt(np.sum(x_diff**2, axis=1)), 
                           np.sqrt(np.sum(y_diff**2, axis=1)))
    
    moranR = numerator / denominator
    
    var = (n**2 * np.einsum('ij,ji', w, w) - 
           2 * n * np.einsum('i,i', np.sum(w, axis=1), np.sum(w, axis=0)) + 
           np.einsum('ij->', w)**2) / (n**2 * (n - 1)**2)
    
    z = moranR / (var**(1/2))
    p = scipy.stats.norm.sf(z)
    
    return moranR, p

In [ ]:
pairs = [(0,1), (1,2), (2,3), (3,4), (4,5), (5,0), (0,2), (0,3), (0,4), (1,3), (1,4), (1,5), (2,4), (2,5), (3,5)]
path = "marker.csv"
df = pd.read_csv(path)
cts = ['GCBC', 'NBC_MBC', 'FDC', 'epithelial', 'CD4_T', 'myeloid']
corrs = []
ps = []

w = moranR_weights(pos, l=1)
#w = KNN.from_array(pos, k=32.5)
#w.transform = "R"

for (m, n) in pairs:
    ct1 = cts[m]
    ct2 = cts[n]
    stged_ct1 = pd.read_csv(f'data/test/stged/{ct1}.csv', index_col=0)
    stged_ct2 = pd.read_csv(f'data/test/stged/{ct2}.csv', index_col=0)
    
    print(f"\n process: {ct1} vs {ct2}")
    mean1 = []
    mean2 = []
    corr = []
    p = []
    for i in range(6):
        gene_indices1 = np.where(np.array(labels[m]) == str(i))[0]
        gene_module1 = np.array(df[f'{ct1}'].dropna().iloc[gene_indices1])
        mean1.append((stged_ct1.loc[gene_module1]).mean(axis=0).values)
        #print(mean1)
    
        gene_indices2 = np.where(np.array(labels[n]) == str(i))[0]
        gene_module2 = np.array(df[f'{ct2}'].dropna().iloc[gene_indices2])  
        mean2.append((stged_ct2.loc[gene_module2]).mean(axis=0).values) 
        #print(mean2)

    mean1 = np.array(mean1)
    mean2 = np.array(mean2)
    moran_I, p_val = moranR(mean1, mean2, w) 
    '''
    for m1 in mean1:
        for m2 in mean2:
            moran_bv = Moran_BV(m1, m2, w)
            corr.append(moran_bv.I)
            p.append(moran_bv.p_sim)
    corr = np.array(corr).reshape(6, 6)
    p = np.array(p).reshape(6,6)
    
    corrs.append(corr)
    ps.append(p)
    '''
    
    corrs.append(moran_I)
    ps.append(p_val)

with open("data/test/moran_p.pkl", "wb") as f:
    pickle.dump((corrs,ps), f)

In [ ]:
pairs = [(0,1), (1,2), (2,3), (3,4), (4,5), (5,0), (0,2), (0,3), (0,4), (1,3), (1,4), (1,5), (2,4), (2,5), (3,5)]
mat = pd.DataFrame(np.zeros((36, 36)))
i = 0
for (x,y) in pairs:
    
    row = x*6
    col = y*6
    mat.iloc[row:row+6, col:col+6] = np.array(C_new[i])
    i = i+1

mat = mat.to_numpy()
mat = np.maximum(mat, mat.T)
cmat = pd.DataFrame(np.ones((36, 36)))

def significance_stars(p_value, value, threshold=0.5):
    if np.isnan(p_value):
        return ''
    if value > threshold:
        if p_value < 0.01:
            return '**'
        elif p_value < 0.05:
            return '*'
    return ''

sig_matrix = np.vectorize(significance_stars)(matrices['combined_p_adj'], mat)

In [ ]:
df = pd.DataFrame(['GCBC']*6+['NBC_MBC']*6+['FDC']*6+['epi']*6+['CD4T']*6+['mye']*6,columns=['cell_type'])
df['cluster'] =[0,1,2,3,4,5]*6
row_ha = pch.HeatmapAnnotation(
    cell_type=pch.anno_simple(df.cell_type, add_text=False, legend=True, cmap='tab20'),
    cluster = pch.anno_simple(df.cluster, add_text =False, legend=False, cmap='tab20'),
    axis=0
)
col_ha = pch.HeatmapAnnotation(
    cell_type=pch.anno_simple(df.cell_type, add_text=False, legend=True, cmap='tab20'),
    cluster = pch.anno_simple(df.cluster, add_text =False, legend=False, cmap='tab20'),
    axis=1
)
plt.figure(figsize=(14, 12))
cm = pch.ClusterMapPlotter(
    #data=pd.DataFrame(mat),
    data=pd.DataFrame(mat),
    top_annotation=col_ha,
    left_annotation=row_ha,
    #row_split=df.cell_type,
    #col_split=df.cell_type, 
    row_cluster=False,
    col_cluster=False,
    #col_dendrogram=True, 
    cmap='Blues',
    vmin=0,
    vmax=mat.max(),
    row_names_side='left',
    #show_rownames=True,
    rasterized=False,
    linewidths=0,
    annot=pd.DataFrame(sig_matrix),
    fmt='',# <-- pass sig_matrix here
    annot_kws={"color":"red", "fontsize":20, "fontweight":"bold"},
    
)

ax = cm.ax_heatmap
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        text = sig_matrix[i, j]
        if text:
            ax.text(j + 0.5, i + 0.5, text,
                   ha='center', va='center',
                   color='red', fontsize=0.2, fontweight='bold',zorder=1001)

#plt.savefig("correlation_heatmap_with_pvalues.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
def plot_single_ct(mat, sig_matrix, cts, ct_idx, n_k=6):
    start = ct_idx * n_k
    end = start + n_k
    
    sub_mat = np.delete(mat[start:end, :], np.s_[start:end], axis=1)
    sub_sig = np.delete(sig_matrix[start:end, :], np.s_[start:end], axis=1)
    
    other_cts = [c for i, c in enumerate(cts) if i != ct_idx]
    col_ct = [c for c in other_cts for _ in range(n_k)]
    col_cluster = list(range(n_k)) * len(other_cts)
    
    df_col = pd.DataFrame({'cell_type': col_ct, 'cluster': col_cluster})
    df_row = pd.DataFrame({'cluster': range(n_k), 'cell_type':cts[ct_idx]})
    
    ct_colors = {
    'GCBC': '#1f77b4',
    'NBC_MBC': '#ff7f0e', 
    'FDC': '#2ca02c',
    'epithelial': '#d62728',
    'CD4_T': '#9467bd',
    'myeloid': '#8c564b'
    }


    cluster_colors = {
    0: '#e41a1c',
    1: '#377eb8',
    2: '#4daf4a',
    3: '#984ea3',
    4: '#ff7f00',
    5: '#a65628'
    }

    col_ha = pch.HeatmapAnnotation(
        cell_type=pch.anno_simple(df_col.cell_type, add_text=False, legend=True, colors=ct_colors, cmap=None),
        cluster=pch.anno_simple(df_col.cluster, add_text=False, legend=False, colors=cluster_colors, cmap=None),
        axis=1
    )
    row_ha = pch.HeatmapAnnotation(
        cell_type=pch.anno_simple(df_row.cell_type, add_text=False, legend=False, colors=ct_colors, cmap=None),
        cluster=pch.anno_simple(df_row.cluster, add_text=False, legend=False, colors=cluster_colors, cmap=None),
        axis=0
    )

    plt.figure(figsize=(12, 2.7))
    cm = pch.ClusterMapPlotter(
        data=pd.DataFrame(sub_mat),
        top_annotation=col_ha,
        left_annotation=row_ha,
        col_split=df_col.cell_type,
        row_cluster=False,
        col_cluster=False,
        cmap='YlGnBu',#'Blues',
        vmin=0,
        vmax=mat.max(),
        rasterized=False,
        linewidths=0,
        annot=pd.DataFrame(sub_sig),
        fmt='',
        annot_kws={"color": "red", "fontsize": 20, "fontweight": "bold"},
        legend_hpad=0,
        legend_vpad=1, 
        label='association'
    )
    ax = cm.ax_heatmap
    ax.text(1.1, 1, '* p < 0.05\n** p < 0.01', 
            transform=ax.transAxes, fontsize=10, color='red',
            fontweight='bold', va='top')
    plt.suptitle(f'{cts[ct_idx]} associations with other cell types', fontsize=14, y=1.02)
    #plt.savefig(f"{cts[ct_idx]}_associations.pdf", dpi=300, bbox_inches="tight")
    plt.show()

plot_single_ct(mat, sig_matrix, cts, ct_idx=5)  # GCBC
plot_single_ct(mat, sig_matrix, cts, ct_idx=1)  # CD4T

In [ ]:
c = np.array(CD4T.iloc[np.where(np.array(CD4_T_label) == '0')[0]])
d = np.array(NBC_MBC.iloc[np.where(np.array(NBC_MBC_label) == '2')[0]])

print(len(c))
print(len(d))

In [ ]:
for i in c:
    print(i)

In [ ]:
for i in d:
    #d = np.array(GCBC.iloc[np.where(np.array(GCBC_label) == f'{i}')[0]])
    print(i)

In [ ]:
adj = torch.load('data/spatial/graph/inter/GCBC_PC.pt',weights_only = False)

In [ ]:
adj.max()

In [ ]:
adj.min()

In [ ]:
pos = np.array(pos)

for l in [0.65, 1.0, 1.5, 2.0, 2.5]:
    w = moranR_weights(pos, l=l)
    n_neighbors = (w > 0.01).sum(axis=1).mean()
    print(f"l={l:.2f} → {n_neighbors:.1f} effective neighbors on average")

In [ ]:
w = moranR_weights(pos, l=2)
w.sum()

In [ ]:
adata = sc.read_h5ad('data/test/tonsil_spatial.h5ad')
adata = adata[adata.obs['donor_id']=='BCLL-8-T']
adata = adata[spots,:].copy()
adata

In [ ]:
dot = pd.read_csv("data/test/dot.csv", index_col=0)
dot = dot.loc[coor.index.intersection(spots)]


adata.obsm["proportion"] = dot.values
adata.uns["cell_type"] = dot.columns.tolist()

In [ ]:
stged_ct = pd.read_csv(f'data/test/stged/NBC_MBC.csv', index_col=0)
cell_types = adata.uns["cell_type"]
idx = cell_types.index('NBC_MBC')
prop = adata.obsm["proportion"][:, idx]
score_matrix = np.zeros((adata.n_obs, 6))
for i in range(6):
    c = np.array(NBC_MBC.iloc[np.where(np.array(NBC_MBC_label) == f'{i}')[0]])
    mean = (stged_ct.loc[c]).mean(axis=0).values
    #mean = adata[:,c].X.mean(axis=1)
    #mean[prop <0.05] = 0
    score_matrix[:, i] = mean.flatten()


pca = PCA(n_components=6)
spot_factor = pca.fit_transform(score_matrix)  # (n_spots, 6)

loadings = pd.DataFrame(
    pca.components_.T,
    #index=[f'CD4T_c{i}' for i in range(6)],
    #columns=[f'PC{i+1}\n({pca.explained_variance_ratio_[i]*100:.1f}%)' for i in range(6)]
)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(loadings, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax)
ax.set_title('NBC_MBC gene modules × factors')
ax.set_ylabel('Gene module (cluster)')
ax.set_xlabel('Factor (PC)')
#plt.savefig(f"NBC_MBC_pca.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(24, 4))
spot_factor_no_pc1 = spot_factor.copy()
spot_factor_no_pc1[:, 0] = 0 
reconstructed = pca.inverse_transform(spot_factor_no_pc1)
for i in range(6):
    adata.obs["residual"] = reconstructed[:, i]
    sc.pl.embedding(adata, color="residual", basis="spatial_rot", cmap="RdBu_r",
                    size=80, vmin = 'p10',ax=axes[i], show=False,
                    title="NBC_MBC gene module mean expression")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
#spot_factor_no_pc1 = spot_factor.copy()
#spot_factor_no_pc1[:, 0] = 0 
#reconstructed = pca.inverse_transform(spot_factor_no_pc1)
#adata.obs["residual"] = reconstructed[:, 2]

stged_ct = pd.read_csv(f'data/test/stged/NBC_MBC.csv', index_col=0)
cell_types = adata.uns["cell_type"]
idx = cell_types.index('NBC_MBC')
prop = adata.obsm["proportion"][:, idx]
score_matrix = np.zeros((adata.n_obs, 6))
c = np.array(NBC_MBC.iloc[np.where(np.array(NBC_MBC_label) == '2')[0]])
mean = (stged_ct.loc[c]).mean(axis=0).values
#mean = adata[:,c].X.mean(axis=1)
#mean[prop <0.05] = 0
adata.obs['mean'] = mean.flatten()
sc.pl.embedding(adata, color="mean", basis="spatial_rot", cmap="RdBu_r",
                size=80, ax=ax, vmin='p85', show=False,
                title="NBC_MBC gene module mean expression")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
#plt.savefig("NBC_MBC_gene_module.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
stged_ct = pd.read_csv(f'data/test/stged/CD4_T.csv', index_col=0)
cell_types = adata.uns["cell_type"]
idx = cell_types.index('CD4_T')
prop = adata.obsm["proportion"][:, idx]
score_matrix = np.zeros((adata.n_obs, 6))
for i in range(6):
    d = np.array(CD4T.iloc[np.where(np.array(CD4_T_label) == f'{i}')[0]])
    mean = (stged_ct.loc[d]).mean(axis=0).values
    #mean = adata[:,d].X.mean(axis=1)
    #mean[prop < 0.05] = 0
    score_matrix[:, i] = mean.flatten()


pca = PCA(n_components=6)
spot_factor = pca.fit_transform(score_matrix)  # (n_spots, 6)

loadings = pd.DataFrame(
    pca.components_.T,
    #index=[f'CD4T_c{i}' for i in range(6)],
    #columns=[f'PC{i+1}\n({pca.explained_variance_ratio_[i]*100:.1f}%)' for i in range(6)]
)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(loadings, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax)
ax.set_title('CD4T gene modules × factors')
ax.set_ylabel('Gene module (cluster)')
ax.set_xlabel('Factor (PC)')
plt.savefig(f"CD4T_pca.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
spot_factor_no_pc1 = spot_factor.copy()
spot_factor_no_pc1[:, 0] = 0 
reconstructed = pca.inverse_transform(spot_factor_no_pc1)
adata.obs["residual"] = reconstructed[:, 0]
sc.pl.embedding(adata, color="residual", basis="spatial_rot", cmap="RdBu_r",
                size=80, ax=ax, show=False,
                title="CD4T gene module mean expression")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("CD4T_gene_module.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(24, 4))
spot_factor_no_pc1 = spot_factor.copy()
spot_factor_no_pc1[:, 0] = 0 
reconstructed = pca.inverse_transform(spot_factor_no_pc1)

for i in range(6):
    adata.obs["residual"] = reconstructed[:, i]
    sc.pl.embedding(adata, color="residual", basis="spatial_rot", cmap="RdBu_r",
                    size=80,ax=axes[i], show=False,
                    title=f"c{i} PC1 removed")

plt.tight_layout()
plt.show()

In [ ]:
stged_ct = pd.read_csv(f'data/test/stged/myeloid.csv', index_col=0)
cell_types = adata.uns["cell_type"]
idx = cell_types.index('myeloid')
prop = adata.obsm["proportion"][:, idx]
score_matrix = np.zeros((adata.n_obs, 6))
for i in range(6):
    e = np.array(mye.iloc[np.where(np.array(myeloid_label) == f'{i}')[0]])
    mean = (stged_ct.loc[e]).mean(axis=0).values
    #mean = adata[:,d].X.mean(axis=1)
    mean[prop == 0] = 0
    score_matrix[:, i] = mean.flatten()


pca = PCA(n_components=6)
spot_factor = pca.fit_transform(score_matrix)  # (n_spots, 6)

loadings = pd.DataFrame(
    pca.components_.T,
    #index=[f'CD4T_c{i}' for i in range(6)],
    #columns=[f'PC{i+1}\n({pca.explained_variance_ratio_[i]*100:.1f}%)' for i in range(6)]
)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(loadings, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax)
ax.set_title('gene modules × factors')
ax.set_ylabel('Gene module (cluster)')
ax.set_xlabel('Factor (PC)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(24, 4))
spot_factor_no_pc1 = spot_factor.copy()
spot_factor_no_pc1[:, 0] = 0 
reconstructed = pca.inverse_transform(spot_factor_no_pc1)
for i in range(6):
    adata.obs["residual"] = reconstructed[:, i]
    sc.pl.embedding(adata, color="residual", basis="spatial_rot", cmap="RdBu_r",
                    size=80, ax=axes[i],vmin =1.3 ,vmax=2.2, show=False,
                    title=f"c{i} PC1 removed")

plt.tight_layout()
plt.show()

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran_BV
import pandas as pd
from libpysal.weights import KNN



w = KNN.from_array(pos, k=8)
#w.transform = "R"
#coords = adata.obsm['spatial']
#w = Queen.from_dataframe(pos)

#moran_bv = Moran_BV(adata.obs['score1'].values, 
 #                    adata.obs['score2'].values, 
  #                   w)
moran_bv = Moran_BV( mean1
                     , mean2,
                     w)
print(f"Bivariate Moran's I: {moran_bv.I}")
print(f"P-value: {moran_bv.p_sim:.5e}")


In [ ]:
matrices

In [ ]:
import numpy as np
from scipy import sparse
from scipy.stats import norm
from tqdm import tqdm
from statsmodels.stats.multitest import multipletests


def load_cross_adj_matrices(folder, cts, pairs):
    import torch
    import os
    
    cross_adj = {}
    
    for (i, j) in pairs:
        name_i = cts[i]
        name_j = cts[j]
        
        path1 = os.path.join(folder, f"{name_i}_{name_j}.pt")
        path2 = os.path.join(folder, f"{name_j}_{name_i}.pt")
        
        if os.path.exists(path1):
            W = torch.load(path1, map_location='cpu')
            if hasattr(W, 'numpy'):
                W = W.numpy()
            cross_adj[(i, j)] = W
            print(f"  Loaded {name_i}_{name_j}.pt  shape={W.shape}")
        elif os.path.exists(path2):
            W = torch.load(path2, map_location='cpu')
            if hasattr(W, 'numpy'):
                W = W.numpy()
            cross_adj[(i, j)] = W.T
            print(f"  Loaded {name_j}_{name_i}.pt (transposed)  shape={W.T.shape}")
        else:
            print(f"  WARNING: {name_i}_{name_j}.pt not found, skipping")
    
    return cross_adj


def compute_association_matrix(W, labels_i, labels_j, n_clusters_i, n_clusters_j):
    
    labels_i = np.asarray(labels_i, dtype=int)
    labels_j = np.asarray(labels_j, dtype=int)
    
    n_i = len(labels_i)
    n_j = len(labels_j)
    
    I_i = np.zeros((n_i, n_clusters_i))
    I_i[np.arange(n_i), labels_i] = 1.0
    
    I_j = np.zeros((n_j, n_clusters_j))
    I_j[np.arange(n_j), labels_j] = 1.0
    
    sizes_i = I_i.sum(axis=0)
    sizes_j = I_j.sum(axis=0)
    
    if sparse.issparse(W):
        W_dense = W.toarray()
    else:
        W_dense = W

    W_binary = (W_dense != 0).astype(float)
    n_edges = I_i.T @ W_binary @ I_j      # (K_i, K_j)

    total_weight = I_i.T @ W_dense @ I_j   # (K_i, K_j)
    

    denom_density = sizes_i[:, None] * sizes_j[None, :]
    denom_density[denom_density == 0] = 1.0
    density = n_edges / denom_density
    
    denom_strength = n_edges.copy()
    denom_strength[denom_strength == 0] = 1.0
    strength = total_weight / denom_strength
    
    return density, strength



def permutation_test_single_pair(W, labels_i, labels_j,
                                  n_clusters_i=6, n_clusters_j=6,
                                  n_permutations=1000, seed=42):
    rng = np.random.RandomState(seed)
    
    obs_density, obs_strength = compute_association_matrix(
        W, labels_i, labels_j, n_clusters_i, n_clusters_j
    )
    
    null_density = np.zeros((n_permutations, n_clusters_i, n_clusters_j))
    null_strength = np.zeros((n_permutations, n_clusters_i, n_clusters_j))
    
    for perm in tqdm(range(n_permutations), desc="Permutation"):
        shuffled_labels_i = rng.permutation(labels_i)
        d, s = compute_association_matrix(
            W, shuffled_labels_i, labels_j, n_clusters_i, n_clusters_j
        )
        null_density[perm] = d
        null_strength[perm] = s
    

    density_null_mean = np.mean(null_density, axis=0)
    density_null_std = np.std(null_density, axis=0)
    density_z = (obs_density - density_null_mean) / (density_null_std + 1e-10)
    
    density_p = np.zeros((n_clusters_i, n_clusters_j))
    for ci in range(n_clusters_i):
        for cj in range(n_clusters_j):
            density_p[ci, cj] = (np.sum(null_density[:, ci, cj] >= obs_density[ci, cj]) + 1) / (n_permutations + 1)
    

    strength_null_mean = np.mean(null_strength, axis=0)
    strength_null_std = np.std(null_strength, axis=0)
    strength_z = (obs_strength - strength_null_mean) / (strength_null_std + 1e-10)
    
    strength_p = np.zeros((n_clusters_i, n_clusters_j))
    for ci in range(n_clusters_i):
        for cj in range(n_clusters_j):
            strength_p[ci, cj] = (np.sum(null_strength[:, ci, cj] >= obs_strength[ci, cj]) + 1) / (n_permutations + 1)
    
    results = {
        'density': {
            'observed': obs_density,
            'null_mean': density_null_mean,
            'null_std': density_null_std,
            'z_scores': density_z,
            'p_values': density_p,
        },
        'strength': {
            'observed': obs_strength,
            'null_mean': strength_null_mean,
            'null_std': strength_null_std,
            'z_scores': strength_z,
            'p_values': strength_p,
        },
        'summary': {
            'n_sig_density': int(np.sum(density_p < 0.05)),
            'n_sig_strength': int(np.sum(strength_p < 0.05)),
            'total': n_clusters_i * n_clusters_j,
        }
    }
    
    return results


def test_all_layer_pairs(cross_adj_matrices, cluster_labels, pairs,
                          n_clusters_per_layer, layer_names=None,
                          n_permutations=1000, seed=42):
    n_layers = len(n_clusters_per_layer)
    if layer_names is None:
        layer_names = [f'Layer_{i}' for i in range(n_layers)]
    
    all_results = {}
    
    print(f"Testing {len(pairs)} layer pairs with {n_permutations} permutations each")
    print("=" * 70)
    
    for (li, lj) in pairs:
        print(f"\n{layer_names[li]} - {layer_names[lj]}:")
        
        W = cross_adj_matrices[(li, lj)]
        result = permutation_test_single_pair(
            W, cluster_labels[li], cluster_labels[lj],
            n_clusters_i=n_clusters_per_layer[li],
            n_clusters_j=n_clusters_per_layer[lj],
            n_permutations=n_permutations,
            seed=seed
        )
        
        all_results[(li, lj)] = result
        s = result['summary']
        print(f"  → density:  {s['n_sig_density']}/{s['total']} significant")
        print(f"  → strength: {s['n_sig_strength']}/{s['total']} significant")
    
    print("\n" + "=" * 70)
    return all_results


# ============================================================
# ============================================================
def results_to_large_matrices(all_results, pairs, n_clusters_per_layer,
                               fdr_alpha=0.05, w_density=0.5):
    total_size = sum(n_clusters_per_layer)
    n_layers = len(n_clusters_per_layer)
    offsets = np.cumsum([0] + list(n_clusters_per_layer))
    w_strength = 1.0 - w_density
    
    mat_density_obs = np.zeros((total_size, total_size))
    mat_strength_obs = np.zeros((total_size, total_size))
    mat_density_z = np.zeros((total_size, total_size))
    mat_strength_z = np.zeros((total_size, total_size))
    mat_density_p = np.ones((total_size, total_size))
    mat_strength_p = np.ones((total_size, total_size))
    
    for (x, y) in pairs:
        if (x, y) not in all_results:
            continue
        rs, re = offsets[x], offsets[x + 1]
        cs, ce = offsets[y], offsets[y + 1]
        
        mat_density_obs[rs:re, cs:ce] = all_results[(x, y)]['density']['observed']
        mat_strength_obs[rs:re, cs:ce] = all_results[(x, y)]['strength']['observed']
        mat_density_z[rs:re, cs:ce] = all_results[(x, y)]['density']['z_scores']
        mat_strength_z[rs:re, cs:ce] = all_results[(x, y)]['strength']['z_scores']
        mat_density_p[rs:re, cs:ce] = all_results[(x, y)]['density']['p_values']
        mat_strength_p[rs:re, cs:ce] = all_results[(x, y)]['strength']['p_values']
    
    mat_density_obs = np.maximum(mat_density_obs, mat_density_obs.T)
    mat_strength_obs = np.maximum(mat_strength_obs, mat_strength_obs.T)
    mat_density_z = np.maximum(mat_density_z, mat_density_z.T)
    mat_strength_z = np.maximum(mat_strength_z, mat_strength_z.T)
    mat_density_p = np.minimum(mat_density_p, mat_density_p.T)
    mat_strength_p = np.minimum(mat_strength_p, mat_strength_p.T)
    
    denom_stouffer = np.sqrt(w_density**2 + w_strength**2)
    mat_combined_z = (w_density * mat_density_z + w_strength * mat_strength_z) / denom_stouffer
    mat_combined_p = 1.0 - norm.cdf(mat_combined_z)
    
    mat_p_adj = np.ones((total_size, total_size))
    mat_significant = np.zeros((total_size, total_size), dtype=bool)
    
    total_sig = 0
    for (x, y) in pairs:
        rs, re = offsets[x], offsets[x + 1]
        cs, ce = offsets[y], offsets[y + 1]
        
        block_p = mat_combined_p[rs:re, cs:ce].flatten()
        reject, p_adj, _, _ = multipletests(block_p, alpha=fdr_alpha, method='fdr_bh')
        
        n_i = re - rs
        n_j = ce - cs
        mat_p_adj[rs:re, cs:ce] = p_adj.reshape((n_i, n_j))
        mat_significant[rs:re, cs:ce] = reject.reshape((n_i, n_j))
        total_sig += int(np.sum(reject))
    
    mat_p_adj = np.minimum(mat_p_adj, mat_p_adj.T)
    mat_significant = mat_significant | mat_significant.T
    
    for i in range(n_layers):
        s, e = offsets[i], offsets[i + 1]
        n = e - s
        mat_density_obs[s:e, s:e] = np.identity(n)
        mat_strength_obs[s:e, s:e] = np.identity(n)
        mat_density_z[s:e, s:e] = 0
        mat_strength_z[s:e, s:e] = 0
        mat_combined_z[s:e, s:e] = 0
        mat_density_p[s:e, s:e] = 1
        mat_strength_p[s:e, s:e] = 1
        mat_combined_p[s:e, s:e] = 1
        mat_p_adj[s:e, s:e] = 1
        mat_significant[s:e, s:e] = False
    
    print(f"Per-block FDR correction (w_density={w_density}): {total_sig} significant across {len(pairs)} blocks")
    
    return {
        'density_observed': mat_density_obs,
        'strength_observed': mat_strength_obs,
        'density_z': mat_density_z,
        'strength_z': mat_strength_z,
        'density_p': mat_density_p,
        'strength_p': mat_strength_p,
        'combined_z': mat_combined_z,
        'combined_p': mat_combined_p,
        'combined_p_adj': mat_p_adj,
        'significant': mat_significant,
    }

pairs = [(0,1), (1,2), (2,3), (3,4), (4,5), (5,0), (0,2), (0,3), (0,4), (1,3), (1,4), (1,5), (2,4), (2,5), (3,5)]
mat = pd.DataFrame(np.zeros((36, 36)))
cts = ['GCBC', 'NBC_MBC', 'FDC', 'epithelial', 'CD4_T', 'myeloid']
cross_adj = load_cross_adj_matrices(
    folder='data/spatial/graph/inter/',
    cts=cts,
    pairs=pairs
)
cluster_labels = labels

results = test_all_layer_pairs(
    cross_adj_matrices=cross_adj,
    cluster_labels=cluster_labels,
    pairs=pairs,
    n_clusters_per_layer=[6, 6, 6, 6, 6, 6],
    layer_names=cts,
    n_permutations=1000
)

matrices = results_to_large_matrices(results, pairs, [6, 6, 6, 6, 6, 6],
                                      fdr_alpha=0.05, w_density=0.9)